<a href="https://colab.research.google.com/github/herlocksholmes1888/llm-text-detection/blob/main/LLM_Detection_Project_30_06_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LLM Text Detector** 🤖
This is a Machine Learning project, created with Kaggle's dataset, made to detect when it's analyzing text that was generated by Artificial Intelligence or when it was created by a Human.

There are a couple of blindspots to be considered, such as the following:

1️⃣ GenAI learned to write texts with human input. There are some human texts that may get a false positive for that reason, especially if the writing has "AI quirks" (extensive usage of —, for example).
2️⃣

## **Project setup** 💻

These steps are necessary to fetch the necessary data for the experiment. Here, we are connected to a Drive with an extensive dataset of human and AI generated texts provided by Kaggle.

In [2]:
# Google Drive connection
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Project imports
import pandas as pd
import torch
from transformers import pipeline

In [ ]:
# Training data
raw = '/content/drive/MyDrive/Human Text vs LLM Text Detection/train_v2_drcat_02.csv'
df = pd.read_csv(raw)

df

,text,label,prompt_name,source,RDizzl3_seven
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False
3,Phones & Driving\n\nDrivers should not be able...,0,Phones and driving,persuade_corpus,False
4,Cell Phone Operation While Driving\n\nThe abil...,0,Phones and driving,persuade_corpus,False
...,...,...,...,...,...
44863,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44864,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44865,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44866,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True


## **LLM text detection**

In [ ]:
# Is there a correlation between `label` and `RDizzl3_seven`?
label_rd_corr = df['label'].corr(df['RDizzl3_seven'])

# There doesn't seem to be any correlation between the two columns
# RDizzl3 appears to be an irrelevant column in the analysis we intend to do
label_rd_corr

np.float64(-0.16283924007362083)

In [ ]:
# Are there any null values?
missing_values = df.isnull().sum()

# There aren't any null values in this dataset
missing_values


,0
text,0
label,0
prompt_name,0
source,0
RDizzl3_seven,0


In [ ]:
# Isolate relevant columns in df
df = df[['text', 'label']]

# For this analysis, only two columns will be necessary: text, which has the content to be analysed, and label, which says
# whether or not the text has been written by a person
df

,text,label
0,Phones\n\nModern humans today are always on th...,0
1,This essay will explain if drivers should or s...,0
2,Driving while the use of cellular devices\n\nT...,0
3,Phones & Driving\n\nDrivers should not be able...,0
4,Cell Phone Operation While Driving\n\nThe abil...,0
...,...,...
44863,"Dear Senator,\n\nI am writing to you today to ...",1
44864,"Dear Senator,\n\nI am writing to you today to ...",1
44865,"Dear Senator,\n\nI am writing to you today to ...",1
44866,"Dear Senator,\n\nI am writing to you today to ...",1


In [ ]:
# What's the size difference between samples?
human_texts_df = df.loc[df['label'] == 0, "text"]
llm_texts_df = df.loc[df['label'] == 1, "text"]

human_texts_length = human_texts_df.str.len()
llm_texts_length = llm_texts_df.str.len()

# How long is the longest text written by a human? How small is the smallest text written by a human?
# What is the average of characters written by a human?
largest_human_text = human_texts_length.max()
average_human_text = human_texts_length.mean()
smallest_human_text = human_texts_length.min()

print(f'Largest human text: {largest_human_text}')
print(f'Smallest human text: {smallest_human_text}')
print(f'Average human text: {average_human_text}')
print('-----------')

# How long is the longest text written by an LLM? How small is the smallest text written by an LLM?
# What is the average of characters written by a LLM?
largest_llm_text = llm_texts_length.max()
average_llm_text = llm_texts_length.mean()
smallest_llm_text = llm_texts_length.min()

print(f'Largest LLM text: {largest_llm_text}')
print(f'Smallest LLM text: {smallest_llm_text}')
print(f'Average LLM text: {average_llm_text}')

# Q: Is there any significant difference between the length of a text written by a human and the length of a text written by an LLM?
# A: Yes. Humans tend to write more words than an LLM.
#
# Are the added words redundant?
# Hypothesis: LLM text tends to be more succint, whilst humans tend to embelish what they write.
# This can be a useful pattern to teach a ML model.

Largest human text: 18322
Smallest human text: 691
Average human text: 2348.503890979504
-----------
Largest LLM text: 5078
Smallest LLM text: 48
Average LLM text: 2009.2924501343086


In [5]:
# What words are more often used by AI, once we ignore filler words?
# What range of emotions tends to be present only when humans write texts?
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# The tokenizer turns words into smaller chunks so that they may be processed by the model
# The model variable is the specific model we're going to use in the analysis;
# think of it like using gpt-5o-mini or llama
model_name = "monologg/bert-base-cased-goemotions-original"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# In order for the next calculations to make sense, we must discover what is the
# mathematical representation of a neutral emotion in the model we are currently using
labels = model.config.id2label
neutral_idx = [idx for idx, label in labels.items() if label.lower() == 'neutral'][0]

# This is an auxiliary function that is meant to do the actual calculation of
# how much emotion there is in a text. We will call it afterward, so that
# we may calculate the emotions of 100 human texts and 100 LLM texts
def calculate_emotion(text):
  # To ensure that the texts do not exceed the model's token limit, the text is
  # truncated once it reaches the limit. Through this line, we are also making
  # it so that we may pass multiple sentences to the model, which is useful to
  # this analysis
  input = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

  # This line of code guarantees that the model is not going through training
  # and therefore does not need to keep track of its mistakes to better itself.
  # In this project, the BERT model is being used; not trained
  #
  # The input is then sent to the neutral network that has been established by
  # the variable models
  with torch.no_grad():
    outputs = model(**input)


# print(f'Percentage of emotions present in human texts: {}')
# print(f'Percentage of emotions present in LLM texts: {}')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

27
